In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='google.protobuf.symbol_database')

import os
import pickle
import cv2
import numpy as np
import mediapipe as mp
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Mediapipe setup
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3)


In [ ]:

# Data directory and containers
DATA_DIR = 'dataset'
data = []
labels = []

In [ ]:
# Process each item in the dataset directory
for dir_ in os.listdir(DATA_DIR):
    dir_path = os.path.join(DATA_DIR, dir_)
    if os.path.isdir(dir_path):  # Check if it is a directory
        for img_path in os.listdir(dir_path):
            data_aux = []
            img = cv2.imread(os.path.join(dir_path, img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Flip the image horizontally
            img_flipped = cv2.flip(img_rgb, 1)
            
            for img_variant in [img_rgb, img_flipped]:  # Use original and flipped images
                data_aux = []
                results = hands.process(img_variant)
                if results.multi_hand_landmarks:
                    for hand_landmarks in results.multi_hand_landmarks:
                        for i in range(len(hand_landmarks.landmark)):
                            x = hand_landmarks.landmark[i].x
                            y = hand_landmarks.landmark[i].y
                            data_aux.append(x)
                            data_aux.append(y)
                    # Ensure the feature vector length is consistent
                    if len(data_aux) == 42:  # 21 landmarks * 2 (x, y)
                        data.append(data_aux)
                        labels.append(dir_)

# Convert labels to numerical format
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

# Save the processed data and label encoder
with open('data.pickle', 'wb') as f:
    pickle.dump({'data': data, 'labels': labels}, f)

with open('label_encoder.pickle', 'wb') as f:
    pickle.dump(label_encoder, f)

In [ ]:

# Load the data
data_dict = pickle.load(open('data.pickle', 'rb'))
data = np.asarray(data_dict['data'])
labels = np.asarray(data_dict['labels'])

# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, shuffle=True, stratify=labels)

# Train the model
model = RandomForestClassifier()
model.fit(x_train, y_train)

# Save the trained model
with open('model.p', 'wb') as f:
    pickle.dump({'model': model}, f)

In [ ]:
# Load the trained model
model_dict = pickle.load(open('model.p', 'rb'))
model = model_dict['model']

# Predict on the test set
y_pred = model.predict(x_test)

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {accuracy * 100:.2f}%')
    print('
Classification Report:')
    print(classification_report(y_test, y_pred))

# Cross-validation
cv_scores = cross_val_score(model, data, labels, cv=5, scoring='accuracy')
print(f'Cross-Validation Accuracy Scores: {cv_scores}')
print(f'Mean Cross-Validation Accuracy: {cv_scores.mean()}')

In [ ]:
# Feature importances from the RandomForestClassifier
importances = model.feature_importances_

# Reshape the importances to correspond to the (x, y) coordinates of the landmarks
importances = np.array(importances).reshape((21, 2))

# Create a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(importances, annot=True, cmap='viridis', xticklabels=['x', 'y'], yticklabels=[f'landmark_{i}' for i in range(21)])
plt.title('Feature Importances Heatmap')
plt.xlabel('Coordinate')
plt.ylabel('Landmark')
plt.show()  

In [ ]:
import cv2
import numpy as np
import pickle

# Load the trained model and label encoder
model_dict = pickle.load(open('model.p', 'rb'))
model = model_dict['model']
label_encoder = pickle.load(open('label_encoder.pickle', 'rb'))

# Mediapipe setup
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(static_image_mode=True, min_detection_confidence=0.3)

# Capture video
cap = cv2.VideoCapture(0)
while True:
    x_ = []
    y_ = []
    data_aux = []
    ret, frame = cap.read()
    
    if not ret:
        print("Failed to capture image from camera. Exiting...")
        break
    
    H, W, _ = frame.shape
    
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)
    
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )
            for i in range(len(hand_landmarks.landmark)):
                x = hand_landmarks.landmark[i].x
                y = hand_landmarks.landmark[i].y
                data_aux.append(x)
                data_aux.append(y)
                x_.append(x)
                y_.append(y)
        
        if len(data_aux) == 42:  # Ensure feature count matches
            x1 = int(min(x_) * W) - 10
            y1 = int(min(y_) * H) - 10
            x2 = int(max(x_) * W) - 10
            y2 = int(max(y_) * H) - 10
            prediction = model.predict([np.asarray(data_aux)])
            predicted_character = label_encoder.inverse_transform([int(prediction[0])])[0]
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), 4)
            cv2.putText(frame, predicted_character, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 0), 3, cv2.LINE_AA)

    cv2.imshow('frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()